### Load original jsons

In [1]:
import json

with open('ptbr-multiple-choice-qa-pairs.json', 'r', encoding='utf-8') as f:
    ptbr_mlc = json.load(f)

with open('ptbr-open-ended-qa-pairs.json', 'r', encoding='utf-8') as f:
    ptbr_oe = json.load(f)

with open('ptpt-multiple-choice-qa-pairs.json', 'r', encoding='utf-8') as f:
    ptpt_mlc = json.load(f)

with open('ptpt-open-ended-qa-pairs.json', 'r', encoding='utf-8') as f:
    ptpt_oe = json.load(f)

### Expand latex figures and tables in ptpt questions

In [2]:
import re
from typing import Dict, List

def expand_references_in_string(
	string: str,
	options: Dict[str, str] | None,
	images: Dict[str, str] | None,
	tables: Dict[str, str] | None,
) -> str:
	options = options or {}
	images = images or {}
	tables = tables or {}

	# Find tokens in parentheses like (image1), (table1), etc.
	tokens = re.findall(r"\(([^)]+)\)", string or "")
	# also extract tokens appearing inside option texts
	if options:
		for opt in options.values():
			tokens.extend(re.findall(r"\(([^)]+)\)", str(opt) or ""))
	# Preserve first-seen order without duplicates
	seen: set[str] = set()
	references: List[str] = []
	for t in tokens:
		if t in seen:
			continue
		if t in images or t in tables:
			seen.add(t)
			references.append(t)

	if not references:
		return string

	parts = [string, "\n\nConteúdo referenciado:"]
    
	for key in references:
		parts.append(f"[{key}]\n{images.get(key) if key in images else tables.get(key, '')}")
	return "\n".join(parts)

### Loop

In [3]:
ptpt_mlc_processed = []
ptpt_oe_processed = []
ptbr_mlc_processed = []
ptbr_oe_processed = []

for item in ptpt_mlc:
    question = item.get("question_verbatim")
    options = item.get("options_verbatim")
    if not question or not options:
        print(f"Skipping item without valid question/options: {item.get('id', 'unknown id')}")
        continue
    expanded_question = expand_references_in_string(
            question,
            options,
            item.get("images"),
            item.get("tables"),
    )
    
    ptpt_mlc_processed.append({
    "problem": expanded_question,
    "choices": options,
    "solution": expand_references_in_string(item.get("answer_verbatim"), {}, item.get("images"), item.get("tables")),
    "answer": item.get("correct_option"),
    "level": item.get("level"),
    "question_has_figure": item.get("contains_latex_figure_in_question"),
    "solution_has_figure": item.get("contains_latex_figure_in_answer"),
    })

for item in ptpt_oe:
    question = item.get("question_verbatim")
    options = {}
    if not question:
        print(f"Skipping item without valid open-ended question: {item.get('id', 'unknown id')}")
        continue
    expanded_question = expand_references_in_string(
            question,
            options,
            item.get("images"),
            item.get("tables"),
    )
    ptpt_oe_processed.append({
    "problem": expanded_question,
    "solution": expand_references_in_string(item.get("answer_verbatim"), {}, item.get("images"), item.get("tables")),
    "level": item.get("level"),
    "question_has_figure": item.get("contains_latex_figure_in_question"),
    "solution_has_figure": item.get("contains_latex_figure_in_answer"),
    })

for item in ptbr_mlc:
    question = item.get("question_verbatim")
    options = item.get("options_verbatim")
    if not question or not options:
        print(f"Skipping item without valid question/options: {item.get('id', 'unknown id')}")
        continue
    # expanded_question = format_options(question, options)
    
    ptbr_mlc_processed.append({
    "problem": question,
    "choices": options,
    "answer": item.get("correct_option"),
    "level": item.get("level"),
    })

for item in ptbr_oe:
    ptbr_oe_processed.append({
    "problem": item.get("question_verbatim"),
    "solution": item.get("answer_verbatim"),
    "level": item.get("level"),
    })

### Print 

In [4]:
""" for entry in ptbr_oe_processed:
    print(f"problem:\n{entry.get('problem','')}\nsolution:\n{entry.get('solution','')}\nanswer:\n{entry.get('answer','')}\nlevel:\n{entry.get('level','')}\n") """

' for entry in ptbr_oe_processed:\n    print(f"problem:\n{entry.get(\'problem\',\'\')}\nsolution:\n{entry.get(\'solution\',\'\')}\nanswer:\n{entry.get(\'answer\',\'\')}\nlevel:\n{entry.get(\'level\',\'\')}\n") '

### Store

In [5]:
with open(f"ptpt_open_ended_test.json", "w", encoding="utf-8") as f:
    json.dump(ptpt_oe_processed, f, ensure_ascii=False, indent=2)

with open(f"ptbr_open_ended_test.json", "w", encoding="utf-8") as f:
    json.dump(ptbr_oe_processed, f, ensure_ascii=False, indent=2)

with open(f"ptpt_multiple_choice_test.json", "w", encoding="utf-8") as f:
    json.dump(ptpt_mlc_processed, f, ensure_ascii=False, indent=2)

with open(f"ptbr_multiple_choice_test.json", "w", encoding="utf-8") as f:
    json.dump(ptbr_mlc_processed, f, ensure_ascii=False, indent=2)